In [ ]:
!pip install numpy pandas opendatasets scikit-learn xgboost --quiet

In [ ]:
import opendatasets as od

In [ ]:
dataset_url = 'https://www.kaggle.com/c/new-york-city-taxi-fare-prediction/overview'

In [ ]:
%%time
od.download(dataset_url)

In [ ]:
data_dir = './new-york-city-taxi-fare-prediction'

In [ ]:
!ls -lh {data_dir}

In [ ]:
!head {data_dir}/train.csv

In [ ]:
!head {data_dir}/test.csv

In [ ]:
!head {data_dir}/sample_submission.csv

In [ ]:
!wc -l {data_dir}/train.csv

In [ ]:
!wc -l {data_dir}/test.csv

In [ ]:
!wc -l {data_dir}/sample_submission.csv

In [ ]:
import pandas as pd
import random

In [ ]:
sample_frac=0.01

In [ ]:
selected_cols='fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count'.split(',')
selected_cols

In [ ]:
dtypes = {
  'fare_amount': 'float32',
 'pickup_longitude': 'float32',
 'pickup_latitude': 'float32',
 'dropoff_longitude':'float32',
 'passenger_count': 'float32'
}

In [ ]:
def skip_row(row_idx):
  if row_idx==0:
    return False
  return random.random()>sample_frac

In [ ]:
random.seed(42)
df = pd.read_csv(data_dir+"/train.csv",
                 usecols=selected_cols,
                 dtype=dtypes,
                 parse_dates=['pickup_datetime'],
                 skiprows=skip_row)

In [ ]:
df

In [ ]:
test_df = pd.read_csv(data_dir+'/test.csv',dtype=dtypes,parse_dates=['pickup_datetime'])

In [ ]:
test_df

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.pickup_datetime.min(),df.pickup_datetime.max()

In [ ]:
test_df.info()

In [ ]:
test_df.describe()

In [ ]:
test_df.pickup_datetime.min(), test_df.pickup_datetime.max()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_df, val_df = train_test_split(df,test_size=0.2,random_state=42)

In [ ]:
len(train_df),len(val_df)

In [ ]:
train_df = train_df.dropna()
val_df = val_df.dropna()

In [ ]:
df.columns

In [ ]:
input_cols = ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count']

In [ ]:
target_col = 'fare_amount'

In [ ]:
train_inputs = train_df[input_cols]

In [ ]:
train_targets = train_df[target_col]

In [ ]:
train_inputs

In [ ]:
train_targets

In [ ]:
val_inputs = val_df[input_cols]
val_targets =  val_df[target_col]

In [ ]:
val_inputs

In [ ]:
val_targets

In [ ]:
test_inputs = test_df[input_cols]

In [ ]:
test_inputs

In [ ]:
import numpy as np

In [ ]:
class MeanRegressor():
  def fit(self, inputs, targets):
    self.mean = targets.mean()
  def predict (self, inputs):
    return np.full(inputs.shape[0],self.mean)

In [ ]:
mean_model = MeanRegressor()

In [ ]:
mean_model.fit(train_inputs,train_targets)

In [ ]:
mean_model.mean

In [ ]:
train_preds = mean_model.predict(train_inputs)

In [ ]:
train_preds

In [ ]:
val_preds = mean_model.predict(val_inputs)

In [ ]:
val_preds

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
mse = mean_squared_error(train_targets,train_preds)
rmse = np.sqrt(mse)
rmse

In [ ]:
val_mse = mean_squared_error(val_targets,val_preds)
val_rmse = np.sqrt(val_mse)
val_rmse

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
linreg_model = LinearRegression()

In [ ]:
linreg_model.fit(train_inputs,train_targets)

In [ ]:
train_preds = linreg_model.predict(train_inputs)

In [ ]:
train_preds

In [ ]:
val_preds = linreg_model.predict(val_inputs)
val_preds

In [ ]:
train_mse = mean_squared_error(train_targets,train_preds)
train_rmse = np.sqrt(train_mse)
train_rmse

In [ ]:
val_mse = mean_squared_error(val_targets,val_preds)
val_rmse = np.sqrt(val_mse)
val_rmse

In [ ]:
test_inputs

In [ ]:
test_preds = linreg_model.predict(test_inputs)

In [ ]:
submission_df = pd.read_csv(data_dir+'/sample_submission.csv')

In [ ]:
submission_df

In [ ]:
def generate_submission_csv(test_preds, fname):
  sub_df = pd.read_csv(data_dir+'/sample_submission.csv')
  sub_df['fare_amount']=test_preds
  sub_df.to_csv(fname,index=False)

In [ ]:
generate_submission_csv(test_preds,'linreg_submission.csv')

In [ ]:
def add_dateparts(df,col):
  df[col+'_year'] = df[col].dt.year
  df[col+'_month'] = df[col].dt.month
  df[col+'_day'] = df[col].dt.day
  df[col+'_weekday'] = df[col].dt.weekday
  df[col+'_hour'] = df[col].dt.hour

In [ ]:
add_dateparts(train_df,'pickup_datetime')

In [ ]:
add_dateparts(val_df,'pickup_datetime')

In [ ]:
add_dateparts(test_df,'pickup_datetime')

In [ ]:
train_df

In [ ]:
import numpy as np
def haversine_np(lon1,lat1,lon2,lat2):
  lon1,lat1,lon2,lat2 = map(np.radians,[lon1,lat1,lon2,lat2])
  dlon = lon2-lon1
  dlat = lat2-lat1
  a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
  c = 2*np.arcsin(np.sqrt(a))
  km = 6367*c
  return km

In [ ]:
def add_trip_distance(df):
  df['trip_distance'] = haversine_np(df['pickup_longitude'],df['pickup_latitude'],df['dropoff_longitude'],df['dropoff_latitude'])

In [ ]:
add_trip_distance(train_df)

In [ ]:
add_trip_distance(val_df)

In [ ]:
add_trip_distance(test_df)

In [ ]:
train_df.sample(5)

In [ ]:
jfk_lonlat = -73.7781, 40.6413
lga_lonlat = -73.8740, 40.7769
ewr_lonlat = -74.1745, 40.6895
met_lonlat = -73.9632, 40.7794
wtc_lonlat = -74.0099, 40.7126

In [ ]:
def add_landmark_dropoff_distance(df,landmark_name, landmark_lonlat):
  lon, lat = landmark_lonlat
  df[landmark_name+'_drop_distance'] = haversine_np(lon,lat,df['dropoff_longitude'],df['dropoff_latitude'])

In [ ]:
%%time
for a_df in [train_df,val_df,test_df]:
  for name , lonlat in [('jfk',jfk_lonlat),('lga',lga_lonlat),('ewr',ewr_lonlat),('met',met_lonlat),('wtc',wtc_lonlat)]:
    add_landmark_dropoff_distance(a_df,name,lonlat)

In [ ]:
train_df.sample(5)

In [ ]:
train_df.describe()

In [ ]:
def remove_outliers(df):
  return df[(df['fare_amount']>=1)&
            (df['fare_amount']<=500)&
            (df['pickup_longitude']>=-75)&
            (df['pickup_longitude']<=-72)&
            (df['dropoff_longitude']>=-75)&
            (df['dropoff_longitude']<=-72)&
            (df['pickup_latitude']>=40)&
            (df['pickup_latitude']<=42)&
            (df['dropoff_latitude']>=40)&
            (df['dropoff_latitude']<=42)&
            (df['passenger_count']>=1)&
            (df['passenger_count']<=6)
            ]

In [ ]:
train_df = remove_outliers(train_df)

In [ ]:
val_df = remove_outliers(val_df)

In [ ]:
train_df.to_parquet('train.parquet')

In [ ]:
val_df.to_parquet('val.parquet')

In [ ]:
test_df.to_parquet('test.parquet')

In [ ]:
train_df.columns

In [ ]:
input_cols = ['pickup_longitude', 'pickup_latitude',
       'dropoff_longitude', 'dropoff_latitude', 'passenger_count',
       'pickup_datetime_year', 'pickup_datetime_month', 'pickup_datetime_day',
       'pickup_datetime_weekday', 'pickup_datetime_hour', 'trip_distance',
       'jfk_drop_distance', 'lga_drop_distance', 'ewr_drop_distance',
       'met_drop_distance', 'wtc_drop_distance']

In [ ]:
target_col = 'fare_amount'

In [ ]:
train_inputs = train_df[input_cols]
train_targets = train_df[target_col]

In [ ]:
val_inputs = val_df[input_cols]
val_targets = val_df[target_col]

In [ ]:
test_inputs = test_df[input_cols]

In [ ]:
def evaluate(model):
  train_preds = model.predict(train_inputs)
  train_mse = mean_squared_error(train_preds,train_targets)
  train_rmse = np.sqrt(train_mse)
  val_preds = model.predict(val_inputs)
  val_mse = mean_squared_error(val_preds,val_targets)
  val_rmse = np.sqrt(val_mse)
  return train_rmse, val_rmse, train_preds, val_preds

In [ ]:
def predict_and_submit(model,fname):
  test_preds = model.predict(test_inputs)
  sub_df = pd.read_csv(data_dir+'/sample_submission.csv')
  sub_df['fare_amount'] = test_preds
  sub_df.to_csv(fname,index=None)
  return sub_df

In [ ]:
from sklearn.linear_model import Ridge

In [ ]:
model1 = Ridge(random_state=42)

In [ ]:
%%time
model1.fit(train_inputs,train_targets)

In [ ]:
evaluate(model1)

In [ ]:
predict_and_submit(model1,'ridge_submission.csv')

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
model2 = RandomForestRegressor(max_depth=10, n_jobs=-1, random_state=42, n_estimators=50)

In [ ]:
%%time
model2.fit(train_inputs,train_targets)

In [ ]:
evaluate(model2)

In [ ]:
predict_and_submit(model2,'rf_submission.csv')

In [ ]:
from xgboost import XGBRegressor

In [ ]:
model3 = XGBRegressor(random_state=42,n_jobs=-1,objective='reg:squarederror')

In [ ]:
%%time
model3.fit(train_inputs,train_targets)

In [ ]:
evaluate(model3)

In [ ]:
predict_and_submit(model3,'xgb_submission.csv')